## 1. Imports and Configuration

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
from datetime import datetime, timedelta


In [3]:
tickers = {
    "Information Technology": ["AAPL", "MSFT", "NVDA", "AVGO", "CRM"],  # High momentum, growth factor exposure
    "Financials": ["JPM", "BAC", "GS", "MS", "BLK", "WFC"],             # Value factor, rate sensitivity
    "Health Care": ["JNJ", "UNH", "LLY", "ABBV", "MRK", "PFE"],         # Defensive, quality factor
    "Consumer Discretionary": ["AMZN", "TSLA", "HD", "MCD", "NKE", "LOW"], # Cyclical, momentum variation
    "Industrials": ["CAT", "HON", "UNP", "RTX", "GE", "DE"],            # Classic value/quality mix
    "Communication Services": ["GOOGL", "META", "DIS", "NFLX", "T"],    # Growth vs. value spread
    "Consumer Staples": ["PG", "KO", "PEP", "WMT", "COST", "CL"],       # Low vol, defensive
    "Energy": ["XOM", "CVX", "COP", "SLB", "EOG"],                      # Value, commodity beta
    "Utilities": ["NEE", "DUK", "SO", "AEP", "EXC"],                    # Low vol, yield factor
    "Real Estate": ["PLD", "AMT", "EQIX", "SPG", "PSA"],                # Yield, rate sensitivity
    "Materials": ["LIN", "APD", "NEM", "FCX", "SHW"],                   # Cyclical, commodity exposure
}

all_tickers = [ticker for sector in tickers.values() for ticker in sector]

In [4]:
data = yf.download(
    [ticker for sector in tickers.values() for ticker in sector],
    start="2015-01-01",
    end="2025-01-01",
    interval = "1mo"
)["Close"]

C:\Users\godwi\AppData\Local\Temp\ipykernel_11252\2185564213.py:1: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(
[*********************100%***********************]  60 of 60 completed


In [5]:
print(data.shape)
# 120 Months x 60 stocks


(120, 60)


In [6]:
# Compute returns
returns = np.log(data).diff().dropna()
print(returns.shape)

(119, 60)


In [7]:
# Data Quality Checks
# 1. Are there any NaNs?
print(returns.isna().sum().sum())

# 2. Are they at the start?
print(returns.iloc[0].isna().sum())

# 3. Flag outliers (returns > 50% or < -50%)
outliers = (returns > 0.5) | (returns < -0.5)
outliers_df = returns[outliers].stack(level = 'Ticker')
print(outliers_df)


0
0
Date        Ticker
2016-02-01  FCX       0.506031
2020-03-01  EOG      -0.565959
            SLB      -0.682553
            SPG      -0.792846
2020-08-01  TSLA      0.554719
2022-04-01  NFLX     -0.676915
dtype: float64


In [8]:
data.to_csv("data/prices.csv")
returns.to_csv("data/returns.csv")

## 2. Data Loading, Universe Construction

 Load prices, compute returns, filter universe

In [9]:
# Compare to benchmark
spy_row = yf.download(
    "SPY",
    start="2015-01-01",
    end="2025-01-01",
    interval = "1mo"
)

spy_prices = spy_row["Close"]["SPY"]
spy_returns = np.log(spy_prices).diff().dropna()

print(spy_returns.isna().sum()) # Check
spy_returns.to_csv("data/spy_returns.csv")


C:\Users\godwi\AppData\Local\Temp\ipykernel_11252\1929563758.py:2: FutureWarning: YF.download() has changed argument auto_adjust default to True
  spy_row = yf.download(
[*********************100%***********************]  1 of 1 completed

0


In [10]:
# Pull Fundamental Data
# Market Cap
shares_dict = {}
for ticker in all_tickers:
    try:
        info = yf.Ticker(ticker).info
        shares = info.get('sharesOutstanding', None)
        shares_dict[ticker] = shares
    except Exception as e:
        print(f"Error fetching data for {ticker}: {e}")
        shares_dict[ticker] = None

shares_series = pd.Series(shares_dict)
missing = shares_series[shares_series.isna()]
print(missing) # None
log_market_cap = np.log(data.multiply(shares_series, axis=1))
log_market_cap.to_csv("data/log_market_cap.csv")

print(log_market_cap.shape) # 120 Months x 60 stocks
print(log_market_cap.head())



Series([], dtype: int64)
(120, 60)
                 AAPL       ABBV        AEP        AMT       AMZN        APD  \
Date                                                                           
2015-01-01  26.665469  24.928175  23.848913  24.259252  25.973893  23.849766   
2015-02-01  26.757547  24.938136  23.761974  24.281589  26.043693  23.919587   
2015-03-01  26.729611  24.905203  23.747128  24.229946  26.022263  23.887966   
2015-04-01  26.735381  25.004636  23.758090  24.233973  26.147584  23.839998   
2015-05-01  26.775554  25.042307  23.747839  24.219740  26.165093  23.862950   

                 AVGO        BAC        BLK        CAT  ...        SLB  \
Date                                                    ...              
2015-01-01  24.326492  25.163941  24.411277  24.043691  ...  25.222113   
2015-02-01  24.541985  25.206584  24.498166  24.087841  ...  25.243369   
2015-03-01  24.536958  25.179659  24.483029  24.052608  ...  25.240596   
2015-04-01  24.457024  25.217267  

In [11]:
from statsmodels.regression.linear_model import OLS
from statsmodels.tools import add_constant
# Check shape
print(returns.shape)        # (T, N)
print(spy_returns.shape)    # (T,)
# Confirm all share the same index
print(returns.index.equals(spy_returns.index))


(119, 60)
(119,)
True


In [12]:
# Compute Raw Signal Values
# 1. Market Beta
# Run OLS over past 36 months
beta_window = 36
monthly_index = returns.index
beta_panel = pd.DataFrame(np.nan, index = monthly_index, columns = all_tickers)

for i, date in enumerate(monthly_index):
    if i < beta_window:
        continue  # Not enough data for the first few months


    window_returns = returns.iloc[i-beta_window:i]
    window_spy = spy_returns.iloc[i-beta_window:i]
    
    valid_mask = window_spy.notna() 
    spy_window_clean = window_spy[valid_mask] # Not needed as already checked
    X = add_constant(spy_window_clean.values) # Add intercept -> Shape (35, 2) [1.0, x_t] Needed to multiply against \beta_0

    for ticker in all_tickers:
        y = window_returns.loc[window_returns.index[valid_mask], ticker]

        if y.isna().sum() > 3: 
            continue

        y_clean = y.fillna(0).values

        try:
            model = OLS(y_clean, X).fit()
            beta_panel.loc[date, ticker] = model.params[1]  # Store the beta coefficient

        except Exception:
            pass

beta_panel.to_csv("data/signal_beta_raw.csv")
print(beta_panel.shape)
    

(119, 60)


In [13]:
# Sanity check
print((beta_panel < -1).sum().sum())
print((beta_panel > 3).sum().sum()) 

0
0


In [ ]:
# 2. Momentum
mom_start = 12 # 12 months lookback
mom_end = 1 # exclude most recent month - noisy
momentum_panel = pd.DataFrame(np.nan, index = monthly_index, columns = all_tickers)

for i, date in enumerate(monthly_index):
    if i < mom_start:
        continue  # Not enough data for the first few months

    window = returns.iloc[i-mom_start:i-mom_end] 
    compounded = (1 + window).prod(axis = 0, skipna = False) - 1
    momentum_panel.loc[date] = compounded

momentum_panel.to_csv("data/signal_momentum_raw.csv")
print(momentum_panel)

                AAPL      MSFT      NVDA      AVGO       CRM       JPM  \
Date                                                                     
2015-02-01       NaN       NaN       NaN       NaN       NaN       NaN   
2015-03-01       NaN       NaN       NaN       NaN       NaN       NaN   
2015-04-01       NaN       NaN       NaN       NaN       NaN       NaN   
2015-05-01       NaN       NaN       NaN       NaN       NaN       NaN   
2015-06-01       NaN       NaN       NaN       NaN       NaN       NaN   
...              ...       ...       ...       ...       ...       ...   
2024-08-01  0.051980  0.314988  1.367512  0.725579  0.082536  0.285725   
2024-09-01  0.159496  0.258990  1.123252  0.687499  0.107280  0.452157   
2024-10-01  0.315452  0.301165  1.478507  0.911244  0.188168  0.553817   
2024-11-01  0.343208  0.257860  1.694839  0.985214  0.294632  0.518054   
2024-12-01  0.176473  0.064544  1.581452  0.789112  0.123137  0.421125   

                 BAC        GS       

In [40]:
# 3. Size
size_panel = log_market_cap.copy()
size_panel = size_panel.reindex(index=monthly_index)
size_panel = size_panel.reindex(columns=all_tickers)

assert size_panel.index.equals(monthly_index), \
    "Index mismatch between size panel and returns index"

assert size_panel.columns.tolist() == all_tickers, \
    "Column mismatch between size panel and returns columns"

size_panel.to_csv("data/signal_size_raw.csv")

In [47]:
# 4. Low Volatility
vol_window = 12
low_vol_panel = -returns.rolling(window = vol_window, min_periods = 10).std() # Negative for low vol = high score
low_vol_panel = low_vol_panel.reindex(index=monthly_index)
low_vol_panel = low_vol_panel.reindex(columns=all_tickers)
low_vol_panel.to_csv("data/signal_lowvol_raw.csv")

In [50]:
# 5. Long-term Reversal
ltr_start = 36
ltr_end = 12
ltr_panel = pd.DataFrame(np.nan, index = monthly_index, columns = all_tickers)

for i, date in enumerate(monthly_index):
    if i < ltr_start:
        continue  
    
    window = returns.iloc[i-ltr_start:i-ltr_end] 
    compounded = (1 + window).prod(axis = 0, skipna = False) - 1
    ltr_panel.loc[date] = -compounded 

ltr_panel.to_csv("data/signal_ltr_raw.csv")

In [51]:
print(beta_panel.shape)
print(momentum_panel.shape)
print(size_panel.shape)
print(low_vol_panel.shape)
print(ltr_panel.shape)

(119, 60)
(119, 60)
(119, 60)
(119, 60)
(119, 60)


In [65]:
# Winsorise, Z-score normalisation and Burn-in
# Set values < 1%ile to 1%ile, > 99%ile to 99%ile, across all stocks, at each time t - hence 'cross sectional'
def winsorise(signal_panel: pd.DataFrame, lower = 0.01, upper = 0.99) -> pd.DataFrame:
    def winsorise_row(row):
        lower_bound = row.quantile(lower)
        upper_bound = row.quantile(upper)
        return row.clip(lower=lower_bound, upper=upper_bound)
    
    return signal_panel.apply(winsorise_row, axis=1)

def normalise(signal_panel: pd.DataFrame) -> pd.DataFrame:
    def normalise_row(row):
        mu = row.mean()
        sig = row.std()
        if sig == 0:
            return row - mu
        return (row - mu) / sig

    return signal_panel.apply(normalise_row, axis=1)

def trim_burn_in(signal_panel: pd.DataFrame, burn_in_periods = 36) -> pd.DataFrame:
    return signal_panel.iloc[burn_in_periods:]

In [66]:
signals_raw = {
    "beta": beta_panel,
    "momentum": momentum_panel,
    "size": size_panel,
    "low_vol": low_vol_panel,
    "ltr": ltr_panel
}

signals_processed = {
    name: trim_burn_in(normalise(winsorise(panel))) for name, panel in signals_raw.items()
}



In [69]:
# Stack into exposure matrix B_t
# Trim signals due to burn-in
for signal in signals_processed.values():
    print(signal.isna().sum().sum())
    print(signal.shape)

# No NaNs


0
(83, 60)
0
(83, 60)
0
(83, 60)
0
(83, 60)
0
(83, 60)


In [73]:
# Stack into exposure matrix B_t 
# equivalent to 60 x 5 slices with t from 0-82

B = np.stack(
    [
        signals_processed["beta"].values,    # shape (83, 60)
        signals_processed["momentum"].values,     # shape (83, 60)
        signals_processed["size"].values,    # shape (83, 60)
        signals_processed["low_vol"].values,   # shape (83, 60)
        signals_processed["ltr"].values   # shape (83, 60)
    ],
    axis=2   # stack along the 3rd dimension
)

# Preserve axis labels as np drops them
dates = signals_processed["beta"].index.tolist()
tickers = signals_processed["beta"].columns.tolist()
factors = list(signals_processed.keys())

print(B.shape)  # (83, 60, 5)
np.save("data/exposure_matrix_B.npy", B)


(83, 60, 5)


## 3. Factor Signal Construction

In [ ]:
# Load signals and import
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

Period-by-period WLS, store factor returns f_t

## 4. PCA on return Panel

### 4.1 Standardise return panel

### 4.2 PCA

### 4.3 Scree plot + Marchenko-Pastur cutoff

### 4.4 Extract loadings B and factor scores f_t

### 4.5 Compare PCA factors vs fundamental factors

## 5. Covariance Matrix Estimation

## 6. Validation and Diagnostics

IC, ICIR, t-stats, residual PCA check

## 7. Results and Visualisations